In [1]:
import joblib
import numpy as np
import pandas as pd
import psycopg2
import pytz
from tqdm import tqdm
from datetime import timedelta
from dateutil.relativedelta import relativedelta

In [2]:
sales = pd.read_csv('pedidos-1743765883268.csv', sep=';', encoding='latin1')

In [3]:
df = sales[['account_id', 'sales_channel_id']].value_counts()
df = pd.DataFrame(df).reset_index()

In [4]:
# EPOCHS = 1000
# BATCH_SIZE = 32
# VALID_SPLIT = 0.1

In [5]:
SAO_PAULO_TZ = pytz.timezone('America/Sao_Paulo')
LOOKBACK = 1
# SEASONAL_PERIODS = (24, 24*7)
# COVERAGE = 0.33
END_DATE = pd.to_datetime(pd.to_datetime(sales['created_date'].max()).strftime("%Y-%m-%d %H:00:00"))
# START_DATE = END_DATE - timedelta(hours=END_DATE.hour)
START_DATE = END_DATE - timedelta(hours=23)
# TEST_SIZE = int((END_DATE - START_DATE).seconds / 60**2 + 1)

In [6]:
# OOT_DATE = START_DATE + timedelta(hours=23)

In [7]:
# OOT_PERIODS = int((OOT_DATE - END_DATE).seconds / 60**2 + 1)

In [8]:
# oot_dates = pd.DatetimeIndex([END_DATE+timedelta(hours=h) for h in range(1, OOT_PERIODS)], freq='h')
# df_oot = pd.DataFrame(index=oot_dates)

In [9]:
df.drop('count', axis=1, inplace=True)
for account_id in df['account_id'].unique():
    df = pd.concat([df, pd.DataFrame({'account_id': [account_id], 'sales_channel_id': ['ALL']})], ignore_index=True)

In [10]:
# df

In [11]:
id_pairs = list(zip(df['account_id'], df['sales_channel_id']))
# id_pairs = list(zip(df_filtered['account_id'], df_filtered['sales_channel_id']))

In [12]:
# weights_chan = {}
# for account_id, sales_channel_id in tqdm(id_pairs):
#     if sales_channel_id == 'ALL':
#         cond = (sales['account_id'] == account_id) & \
#                (sales['status'].notna())
#     else:
#         cond = (sales['account_id'] == account_id) & \
#                (sales['sales_channel_id'] == sales_channel_id) & \
#                (sales['status'].notna())

#     df_client = sales[cond].drop(['account_id', 'sales_channel_id'], axis=1)
#     df_client['created_date'] = pd.to_datetime(df_client['created_date'], format='%Y-%m-%d %H:%M:%S.%f %z')
#     df_client = df_client.sort_values('created_date').reset_index(drop=True)
#     df_client['created_date'] = df_client['created_date'].dt.strftime("%Y-%m-%d %H:00:00").reset_index(drop=True)

#     df_client_mod = df_client.groupby('created_date').agg(price_total_agg=('price_total', 'sum'), n_orders=('created_date', 'count'))
#     df_client_mod.index = pd.to_datetime(df_client_mod.index, format='%Y-%m-%d %H:00:00')

#     end_date = pd.to_datetime(END_DATE, format='%Y-%m-%d %H:%M:%S')
#     start_date = end_date - relativedelta(months=LOOKBACK)
#     date_index = pd.Series(pd.date_range(start=start_date, end=end_date, freq='h', name='created_date'))
#     df_client_mod = pd.merge(date_index, df_client_mod, how='left', on='created_date').set_index('created_date')
    
#     weights_chan[account_id] = {} if account_id not in weights_chan else weights_chan[account_id]
    
#     df = df_client_mod['n_orders'].fillna(0)
#     df = df.loc[df.index < START_DATE].copy()
    
#     weights_chan[account_id][sales_channel_id] = df.median()

In [13]:
# weights_df = pd.DataFrame(weights_chan)

# weights_df

In [14]:
# weights_df = weights_df.stack().reset_index().rename(columns={'level_0': 'sales_channel_id', 'level_1': 'account_id', 0: 'weights'}).sort_values(by=['account_id', 'sales_channel_id'])

# weights_df

In [15]:
# weights_chan_df = weights_df.groupby('account_id')['weights'].apply(lambda x: x / x.sum()).fillna(0).reset_index().rename(columns={0: 'weights'}).drop('level_1', axis=1)

# weights_chan_df = pd.concat([weights_df['sales_channel_id'].reset_index(drop=True), weights_chan_df], axis=1)

# weights_chan_df

In [16]:
# weights_acc_df = weights_df.groupby('account_id')['weights'].sum() / weights_df.groupby('account_id')['weights'].sum().sum()

# weights_acc_df

In [17]:
# weights_acc_df.sort_values(ascending=False)

In [18]:
# joblib.dump(weights_chan_df, 'weights_chan_df.pkl')
# joblib.dump(weights_acc_df, 'weights_acc_df.pkl')

weights_chan_df = joblib.load('weights_chan_df.pkl')
weights_acc_df = joblib.load('weights_acc_df.pkl')

In [19]:
dbname = 'railway'
username = 'sinatra'
pwd = '781B3XjpeuqE'
hostname = 'monorail.proxy.rlwy.net'
port = 25096

connection = psycopg2.connect(database=dbname, user=username, password=pwd, host=hostname, port=port)
cursor = connection.cursor()

In [20]:
cursor.execute("select distinct account_id, channel from public.forecast where model='TBATS';")
res_tbats = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='GradientBoosting';")
res_gb = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='LSTM';")
res_lstm = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='Ensemble';")
res_ens = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)';")
res_chronos1 = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='Chronos';")
res_chronos2 = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='MEDIAN_MAD_60';")
res_median = cursor.fetchall()

In [21]:
id_pairs = (
    set(res_tbats)
    .intersection(set(res_gb))
    .intersection(set(res_lstm))
    .intersection(set(res_ens))
    .intersection(set(res_chronos1))
    .intersection(set(res_chronos2))
    .intersection(set(res_median))
)


# id_pairs = set(res_tbats).intersection(set(res_gb))

In [22]:
id_pairs

{('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '1'),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '3'),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '5'),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '6'),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', 'ALL'),
 ('0661e926-f219-456e-bde5-612fee6f891c', '1'),
 ('0661e926-f219-456e-bde5-612fee6f891c', 'ALL'),
 ('0ad2f65b-ee0e-4d50-93e2-84b76282325f', '1'),
 ('0ad2f65b-ee0e-4d50-93e2-84b76282325f', '2'),
 ('0ad2f65b-ee0e-4d50-93e2-84b76282325f', 'ALL'),
 ('173e2b27-8bc6-44d1-b581-a1913d6b1894', '1'),
 ('173e2b27-8bc6-44d1-b581-a1913d6b1894', '11'),
 ('173e2b27-8bc6-44d1-b581-a1913d6b1894', '14'),
 ('173e2b27-8bc6-44d1-b581-a1913d6b1894', '16'),
 ('173e2b27-8bc6-44d1-b581-a1913d6b1894', '18'),
 ('173e2b27-8bc6-44d1-b581-a1913d6b1894', '5'),
 ('173e2b27-8bc6-44d1-b581-a1913d6b1894', '8'),
 ('173e2b27-8bc6-44d1-b581-a1913d6b1894', '9'),
 ('173e2b27-8bc6-44d1-b581-a1913d6b1894', 'ALL'),
 ('2058f680-aace-4b31-aed4-f34397b75096', '1'),
 ('2058f680-aace-4b31-aed4-f

In [23]:
len(id_pairs)

156

In [24]:
# gb_models = [f'GradientBoosting_{int(10*n)}' for n in range(1, 10)]
# lstm_models = [f'LSTM_{int(10*n)}' for n in range(1, 10)]
# tbats_models = [f'TBATS_{int(10*n)}' for n in range(1, 10)]
# ens_models = [f'Ensemble_{int(10*n)}' for n in range(1, 10)]
# chronos_models = [f'Chronos_{int(10*n)}' for n in range(1, 10)]

In [25]:
df_forecast_metrics1_norm, df_forecast_metrics2_norm = {}, {}
scores11, scores12 = {}, {}
# for model in gb_models + lstm_models + tbats_models + ens_models + chronos_models:
for model in ['GradientBoosting', 'LSTM', 'TBATS', 'Ensemble', 'Chronos']:
    print('\n\n')
    print(model)
    df_forecast_metrics1_norm[model], df_forecast_metrics2_norm[model]= {}, {}
    
    for account_id, sales_channel_id in id_pairs:
        
        print(account_id, sales_channel_id)
        
        if sales_channel_id == 'ALL':
            cond = (sales['account_id'] == account_id) & \
                   (sales['status'].notna())
        else:
            sales_channel_id = int(sales_channel_id)
            cond = (sales['account_id'] == account_id) & \
                   (sales['sales_channel_id'] == sales_channel_id) & \
                   (sales['status'].notna())

        df_client = sales[cond].drop(['account_id', 'sales_channel_id'], axis=1)
        df_client['created_date'] = pd.to_datetime(df_client['created_date'], format='%Y-%m-%d %H:%M:%S.%f %z')
        df_client = df_client.sort_values('created_date').reset_index(drop=True)
        df_client['created_date'] = df_client['created_date'].dt.strftime("%Y-%m-%d %H:00:00").reset_index(drop=True)

        df_client_mod = df_client.groupby('created_date').agg(price_total_agg=('price_total', 'sum'), n_orders=('created_date', 'count'))
        df_client_mod.index = pd.to_datetime(df_client_mod.index, format='%Y-%m-%d %H:00:00')

        end_date = pd.to_datetime(END_DATE, format='%Y-%m-%d %H:%M:%S')
        start_date = end_date - relativedelta(months=LOOKBACK)
        date_index = pd.Series(pd.date_range(start=start_date, end=end_date, freq='h', name='created_date'))
        df_client_mod = pd.merge(date_index, df_client_mod, how='left', on='created_date').set_index('created_date')
        
        df_forecast_metrics1_norm[model][account_id] = {} if account_id not in df_forecast_metrics1_norm[model] else df_forecast_metrics1_norm[model][account_id]
        df_forecast_metrics2_norm[model][account_id] = {} if account_id not in df_forecast_metrics2_norm[model] else df_forecast_metrics2_norm[model][account_id]
        for dataset in ['sales', 'orders']:
            if dataset == 'sales':
                col = 'price_total_agg'
            else:
                col = 'n_orders'
            
            df = df_client_mod[col].fillna(0)
            
            y_test = df.loc[(df.index >= START_DATE) & (df.index <= END_DATE)].copy()
        
            cursor.execute(f"select start, {dataset}_high, {dataset}_low, {dataset}_mean from public.forecast where account_id = '{account_id}' and channel = '{sales_channel_id}' and model = '{model}'")
            res = cursor.fetchall()
            
            df_forecast = pd.DataFrame(res, columns=['start', f'{dataset}_high', f'{dataset}_low', f'{dataset}_mean']) #[:len(y_test)].set_index(y_test.index)
            
            df_forecast[f'{dataset}_mean'].index = y_test.index
            df_forecast[f'{dataset}_low'].index = y_test.index
            df_forecast[f'{dataset}_high'].index = y_test.index
            
            forecast_mean = df_forecast[f'{dataset}_mean']
            forecast_low = df_forecast[f'{dataset}_low']
            forecast_high = df_forecast[f'{dataset}_high']
            
            forecast = pd.concat([y_test, forecast_mean, forecast_low, forecast_high], axis=1)
            forecast.columns = ['actual', 'forecast', 'lower', 'upper']
            
            df_forecast = forecast.assign(
                covered_pts1=lambda x:
                    4*x['actual'].between(x['lower'], x['upper'], inclusive='both') +
                    2*(x['actual'].between(2*x['lower']-x['forecast'], x['lower'], inclusive='left') + x['actual'].between(x['upper'], 2*x['upper']-x['forecast'], inclusive='right')) +
                    1*(x['actual'].between(3*x['lower']-2*x['forecast'], 2*x['lower']-x['forecast'], inclusive='left') + x['actual'].between(2*x['upper']-x['forecast'], 3*x['upper']-2*x['forecast'], inclusive='right')),
                covered_pts2=lambda x: x['actual'].between(3*x['lower']-2*x['forecast'], 3*x['upper']-2*x['forecast'], inclusive='both'),
                covered_width=lambda x: x['upper'] - x['lower']
            )
            
            df_forecast_metrics = {}
            df_forecast_metrics['total_covered1'] = df_forecast['covered_pts1'].sum()
            df_forecast_metrics['median_covered1'] = df_forecast['covered_pts1'].median()
            df_forecast_metrics['total_covered2'] = df_forecast['covered_pts2'].sum()
            df_forecast_metrics['median_covered2'] = df_forecast['covered_pts2'].median()
            df_forecast_metrics['median_covered_width'] = df_forecast['covered_width'].median()
            
            if dataset == 'orders':
                df_forecast_orders_metrics1_norm = df_forecast_metrics['median_covered1'] / (1 + np.log(1+df_forecast_metrics['median_covered_width']))
                df_forecast_orders_metrics2_norm = 10*df_forecast_metrics['median_covered2'] / (1 + np.log(1+df_forecast_metrics['median_covered_width']))
            else:
                df_forecast_sales_metrics1_norm = df_forecast_metrics['median_covered1'] / (1 + np.log(1+df_forecast_metrics['median_covered_width']))
                df_forecast_sales_metrics2_norm = 10*df_forecast_metrics['median_covered2'] / (1 + np.log(1+df_forecast_metrics['median_covered_width']))
        
        weight_chan = weights_chan_df.loc[(weights_chan_df['account_id'] == account_id) & (weights_chan_df['sales_channel_id'] == sales_channel_id), 'weights'].values[0]
        
        df_forecast_metrics1_norm[model][account_id][sales_channel_id] = weight_chan * (1/3*df_forecast_sales_metrics1_norm + 2/3*df_forecast_orders_metrics1_norm)
        # df_forecast_metrics1_norm[model][(account_id, sales_channel_id)] = weight_chan * (1/3*df_forecast_sales_metrics1_norm + 2/3*df_forecast_sales_metrics2_norm)
        
        df_forecast_metrics2_norm[model][account_id][sales_channel_id] = weight_chan * (1/3*df_forecast_orders_metrics2_norm + 2/3*df_forecast_sales_metrics2_norm)
        # df_forecast_metrics2_norm[model][(account_id, sales_channel_id)] = weight_chan * (1/3*df_forecast_orders_metrics2_norm + 2/3*df_forecast_sales_metrics2_norm)
        
    scores11[model] = np.sum([weights_acc_df[acc_id] * np.sum(list(df_forecast_metrics1_norm[model][acc_id].values())) for acc_id in df_forecast_metrics1_norm[model].keys()])
    scores12[model] = np.sum([weights_acc_df[acc_id] * np.sum(list(df_forecast_metrics2_norm[model][acc_id].values())) for acc_id in df_forecast_metrics2_norm[model].keys()])
    
    print(scores11[model], scores12[model]) # ()




GradientBoosting
f9129295-bb08-4330-b60c-9f0beadda521 32
60ca8452-5b14-481f-a637-8756f527aee8 10
775dcad4-0733-4c22-8dbd-cd3ce344e891 ALL
05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6 6
ca35e88d-d191-4722-b90c-92f4a249869b 7
f9129295-bb08-4330-b60c-9f0beadda521 8
775dcad4-0733-4c22-8dbd-cd3ce344e891 2
9d1296ca-9e4f-4840-b7b5-e12172acab78 1
9d1296ca-9e4f-4840-b7b5-e12172acab78 ALL
173e2b27-8bc6-44d1-b581-a1913d6b1894 9
f9129295-bb08-4330-b60c-9f0beadda521 34
f9129295-bb08-4330-b60c-9f0beadda521 6
6276b87c-ebb2-11ed-a05b-0242ac120003 8
74abffac-6542-4eff-8eae-cc85463a5d02 1
f9129295-bb08-4330-b60c-9f0beadda521 33
92f116fe-5818-41db-9eb5-87445ad3818e ALL
f9129295-bb08-4330-b60c-9f0beadda521 10
e2de40dd-743b-4105-9077-3f343fea980d ALL
6cd2fa8a-5bac-4cc0-b4df-bd2c7d29e71b 1
6276b87c-ebb2-11ed-a05b-0242ac120003 6
92f116fe-5818-41db-9eb5-87445ad3818e 2
7412074f-0f3a-4724-b035-7fb4db3174c8 1
22fde42e-45d7-46cd-9d6f-3a4dbabbc579 ALL
d639773a-df2c-421e-ac87-6918a754572f ALL
8fa00df1-38eb-426f-81a6-118

In [26]:
scores11

{'GradientBoosting': np.float64(0.638228483593443),
 'LSTM': np.float64(0.6095422158598462),
 'TBATS': np.float64(0.6902190187404204),
 'Ensemble': np.float64(0.6324611185319894),
 'Chronos': np.float64(0.6794345245122004)}

In [27]:
scores12

{'GradientBoosting': np.float64(1.3326538805840833),
 'LSTM': np.float64(1.3673718427337063),
 'TBATS': np.float64(1.3854279165077013),
 'Ensemble': np.float64(1.3135520820594917),
 'Chronos': np.float64(1.3637737101410874)}

In [28]:
joblib.dump(pd.DataFrame(scores11, columns=['weights']), 'scores11.pkl')
joblib.dump(pd.DataFrame(scores12, columns=['weights']), 'scores12.pkl');

In [29]:
scores12

{'GradientBoosting': np.float64(1.3326538805840833),
 'LSTM': np.float64(1.3673718427337063),
 'TBATS': np.float64(1.3854279165077013),
 'Ensemble': np.float64(1.3135520820594917),
 'Chronos': np.float64(1.3637737101410874)}

In [42]:
scores11 = joblib.load('scores11.pkl')
scores12 = joblib.load('scores12.pkl')
scores21 = joblib.load('scores21.pkl')
scores22 = joblib.load('scores22.pkl')

In [32]:
scores1 = pd.concat([scores11, scores21], axis=0)
joblib.dump(scores1, 'scores1.pkl')
scores1.to_csv('scores1.csv', sep=';', encoding='latin1', index=True)

scores1

/var/folders/k1/3tn1t6gd2hn967rvpqnz2k8m0000gp/T/ipykernel_82404/710657758.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  scores1 = pd.concat([scores11, scores21], axis=0)


,weights
"amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)",0.438251
MEDIAN_MAD_60,0.645104


In [33]:
scores2 = pd.concat([scores12, scores22], axis=0).sort_values('weights', ascending=False)
joblib.dump(scores2, 'scores2.pkl')
scores2.to_csv('scores2.csv', sep=';', encoding='latin1', index=True)

scores2

/var/folders/k1/3tn1t6gd2hn967rvpqnz2k8m0000gp/T/ipykernel_82404/562971219.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  scores2 = pd.concat([scores12, scores22], axis=0).sort_values('weights', ascending=False)


,weights
"amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)",1.472819
MEDIAN_MAD_60,1.289020


In [34]:
scores1.to_csv('scores1_overall.csv', sep=';', encoding='latin1', index=True)
scores2.to_csv('scores2_overall.csv', sep=';', encoding='latin1', index=True)

In [35]:
df11 = pd.DataFrame(df_forecast_metrics1_norm)
joblib.dump(df11, 'df_scores11.pkl')

df11

,GradientBoosting,LSTM,TBATS,Ensemble,Chronos
f9129295-bb08-4330-b60c-9f0beadda521,"{32: 0.0, 8: 0.0, 34: 0.0, 6: 0.0, 33: 0.0, 10...","{32: 0.0, 8: 0.0, 34: 0.0, 6: 0.0, 33: 0.0, 10...","{32: 0.0, 8: 0.0, 34: 0.0, 6: 0.0, 33: 0.0, 10...","{32: 0.0, 8: 0.0, 34: 0.0, 6: 0.0, 33: 0.0, 10...","{32: 0.0, 8: 0.0, 34: 0.0, 6: 0.0, 33: 0.0, 10..."
60ca8452-5b14-481f-a637-8756f527aee8,"{10: 0.0, 1: 0.0, 'ALL': 0.0, 6: 0.0}","{10: 0.0, 1: 0.0, 'ALL': 0.0, 6: 0.0}","{10: 0.0, 1: 0.0, 'ALL': 0.0, 6: 0.0}","{10: 0.0, 1: 0.0, 'ALL': 0.0, 6: 0.0}","{10: 0.0, 1: 0.0, 'ALL': 0.0, 6: 0.0}"
775dcad4-0733-4c22-8dbd-cd3ce344e891,"{'ALL': 0.33556436846803506, 2: 0.629455854981...","{'ALL': 0.7348391676372723, 2: 0.4129745846807...","{'ALL': 0.32197191149112003, 2: 0.321198867643...","{'ALL': 0.5033465527020526, 2: 0.3362415513946...","{'ALL': 0.5365594604731142, 2: 0.5256906085751..."
05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,"{6: 0.14205120538873522, 3: 0.1957322293222127...","{6: 0.15189793773911958, 3: 0.2077912012187713...","{6: 0.1573731734328607, 3: 0.21148822909678192...","{6: 0.14205120538873522, 3: 0.1957322284375925...","{6: 0.144343256721419, 3: 0.19096455635730825,..."
ca35e88d-d191-4722-b90c-92f4a249869b,"{7: 0.0, 'ALL': 0.7842858343015582, 12: 0.0, 1...","{7: 0.0, 'ALL': 0.5171981435994957, 12: 0.0, 1...","{7: 0.0, 'ALL': 0.8052413948701183, 12: 0.0, 1...","{7: 0.0, 'ALL': 0.47486362649823854, 12: 0.0, ...","{7: 0.0, 'ALL': 0.6123338761715464, 12: 0.0, 1..."
9d1296ca-9e4f-4840-b7b5-e12172acab78,"{1: 0.0, 'ALL': 0.0, 4: 0.0, 5: 0.0, 9: 0.0}","{1: 0.0, 'ALL': 0.0, 4: 0.0, 5: 0.0, 9: 0.0}","{1: 0.0, 'ALL': 0.0, 4: 0.0, 5: 0.0, 9: 0.0}","{1: 0.0, 'ALL': 0.0, 4: 0.0, 5: 0.0, 9: 0.0}","{1: 0.0, 'ALL': 0.0, 4: 0.0, 5: 0.0, 9: 0.0}"
173e2b27-8bc6-44d1-b581-a1913d6b1894,"{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.2...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.3...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.2...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.4...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.3..."
6276b87c-ebb2-11ed-a05b-0242ac120003,"{8: 0.18326106533246228, 6: 0.0, 9: 0.15578382...","{8: 0.22696885595503102, 6: 0.0, 9: 0.12257260...","{8: 0.3487791709164587, 6: 0.0, 9: 0.184552312...","{8: 0.19581169661613199, 6: 0.0, 9: 0.19894295...","{8: 0.28654914517245184, 6: 0.0, 9: 0.26342971..."
74abffac-6542-4eff-8eae-cc85463a5d02,"{1: 0.42691774830664275, 2: 0.0, 'ALL': 0.4268...","{1: 0.25171724264465, 2: 0.0, 'ALL': 0.2516622...","{1: 0.361879115767993, 2: 0.0, 'ALL': 0.361172...","{1: 0.33008308864706254, 2: 0.0, 'ALL': 0.4268...","{1: 0.38885693337925126, 2: 0.0, 'ALL': 0.3878..."
92f116fe-5818-41db-9eb5-87445ad3818e,"{'ALL': 0.5687349620042466, 2: 0.0, 5: 0.37920...","{'ALL': 0.41483250622501744, 2: 0.0, 5: 0.4148...","{'ALL': 0.717576456100309, 2: 0.0, 5: 0.717390...","{'ALL': 0.3791566413361644, 2: 0.0, 5: 0.11718...","{'ALL': 0.5522120082303189, 2: 0.0, 5: 0.28910..."


In [36]:
df12 = pd.DataFrame(df_forecast_metrics2_norm)
joblib.dump(df12, 'df_scores12.pkl')

df12

,GradientBoosting,LSTM,TBATS,Ensemble,Chronos
f9129295-bb08-4330-b60c-9f0beadda521,"{32: 0.0, 8: 0.0, 34: 0.0, 6: 0.0, 33: 0.0, 10...","{32: 0.0, 8: 0.0, 34: 0.0, 6: 0.0, 33: 0.0, 10...","{32: 0.0, 8: 0.0, 34: 0.0, 6: 0.0, 33: 0.0, 10...","{32: 0.0, 8: 0.0, 34: 0.0, 6: 0.0, 33: 0.0, 10...","{32: 0.0, 8: 0.0, 34: 0.0, 6: 0.0, 33: 0.0, 10..."
60ca8452-5b14-481f-a637-8756f527aee8,"{10: 0.0, 1: 0.0, 'ALL': 0.0, 6: 0.0}","{10: 0.0, 1: 0.0, 'ALL': 0.0, 6: 0.0}","{10: 0.0, 1: 0.0, 'ALL': 0.0, 6: 0.0}","{10: 0.0, 1: 0.0, 'ALL': 0.0, 6: 0.0}","{10: 0.0, 1: 0.0, 'ALL': 0.0, 6: 0.0}"
775dcad4-0733-4c22-8dbd-cd3ce344e891,"{'ALL': 1.1614784508165195, 2: 1.1633076055101...","{'ALL': 1.2544550119230111, 2: 1.2564723188462...","{'ALL': 1.11784757176871, 2: 1.115745022162797...","{'ALL': 1.1614784508165195, 2: 1.1633076957544...","{'ALL': 1.08108796822501, 2: 0.657113260718985..."
05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,"{6: 0.2687836755945146, 3: 0.3791340371395408,...","{6: 0.2840297365726457, 3: 0.39831999423898784...","{6: 0.28887472015507765, 3: 0.3998031356637399...","{6: 0.2687836755945146, 3: 0.37913403603376566...","{6: 0.27765032825534464, 3: 0.4106718769057024..."
ca35e88d-d191-4722-b90c-92f4a249869b,"{7: 0.0, 'ALL': 1.4628949108443339, 12: 0.0, 1...","{7: 0.0, 'ALL': 1.5809377284690926, 12: 0.0, 1...","{7: 0.0, 'ALL': 1.405972679101682, 12: 0.0, 1:...","{7: 0.0, 'ALL': 1.4628948479412456, 12: 0.0, 1...","{7: 0.0, 'ALL': 1.2801628784544734, 12: 0.0, 1..."
9d1296ca-9e4f-4840-b7b5-e12172acab78,"{1: 0.0, 'ALL': 0.0, 4: 0.0, 5: 0.0, 9: 0.0}","{1: 0.0, 'ALL': 0.0, 4: 0.0, 5: 0.0, 9: 0.0}","{1: 0.0, 'ALL': 0.0, 4: 0.0, 5: 0.0, 9: 0.0}","{1: 0.0, 'ALL': 0.0, 4: 0.0, 5: 0.0, 9: 0.0}","{1: 0.0, 'ALL': 0.0, 4: 0.0, 5: 0.0, 9: 0.0}"
173e2b27-8bc6-44d1-b581-a1913d6b1894,"{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.9...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.9...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.8...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.9...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.8..."
6276b87c-ebb2-11ed-a05b-0242ac120003,"{8: 0.6464121325862014, 6: 0.0, 9: 0.694655501...","{8: 0.6990106365337003, 6: 0.0, 9: 0.751080449...","{8: 0.6173376617776747, 6: 0.0, 9: 0.650031458...","{8: 0.6464121325862014, 6: 0.0, 9: 0.694655501...","{8: 0.5423233660326015, 6: 0.0, 9: 0.548432195..."
74abffac-6542-4eff-8eae-cc85463a5d02,"{1: 0.8799645234114103, 2: 0.0, 'ALL': 0.87984...","{1: 0.9374031652391575, 2: 0.0, 'ALL': 0.93726...","{1: 0.9405630602381798, 2: 0.0, 'ALL': 0.93931...","{1: 0.8799645172956823, 2: 0.0, 'ALL': 0.87984...","{1: 0.8644920553079056, 2: 0.0, 'ALL': 0.88037..."
92f116fe-5818-41db-9eb5-87445ad3818e,"{'ALL': 1.2836928467452888, 2: 0.0, 5: 1.28381...","{'ALL': 1.3873551053659365, 2: 0.0, 5: 1.38748...","{'ALL': 1.2182090765356826, 2: 0.0, 5: 1.21790...","{'ALL': 1.2836928467452888, 2: 0.0, 5: 0.86577...","{'ALL': 1.069847067375283, 2: 0.0, 5: 1.093341..."


In [37]:
df_scores11 = joblib.load('df_scores11.pkl')
df_scores12 = joblib.load('df_scores12.pkl')
df_scores21 = joblib.load('df_scores21.pkl')
df_scores22 = joblib.load('df_scores22.pkl')

In [38]:
df_scores1 = df_scores11.merge(df_scores21, how='outer', left_index=True, right_index=True, sort=False)
joblib.dump(df_scores1, 'df_scores1.pkl')
df_scores1.to_csv('df_scores1.csv', sep=';', encoding='latin1', index=True)

df_scores1

,GradientBoosting,LSTM,TBATS,Ensemble,Chronos,"amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)",MEDIAN_MAD_60
05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,"{6: 0.14205120538873522, 3: 0.1957322293222127...","{6: 0.15189793773911958, 3: 0.2077912012187713...","{6: 0.1573731734328607, 3: 0.21148822909678192...","{6: 0.14205120538873522, 3: 0.1957322284375925...","{6: 0.144343256721419, 3: 0.19096455635730825,...","{6: 0.0974936067543283, 'ALL': 0.3818834440712...","{6: 0.13581602876671742, 'ALL': 0.315320889948..."
0661e926-f219-456e-bde5-612fee6f891c,"{1: 0.5097794107374761, 'ALL': 0.5097794107374...","{1: 0.5098890573973827, 'ALL': 0.5098890575889...","{1: 0.40382837513277886, 'ALL': 0.403828375132...","{1: 0.5097793943287842, 'ALL': 0.5097793943287...","{1: 0.5226197237158193, 'ALL': 0.5523754794596...","{'ALL': 0.24230124561810834, 1: 0.297588137544...","{'ALL': 0.10575993245413426, 1: 0.041990600614..."
0ad2f65b-ee0e-4d50-93e2-84b76282325f,"{'ALL': 0.4500724264082394, 2: 0.0, 1: 0.31166...","{'ALL': 0.3502143586396694, 2: 0.0, 1: 0.33212...","{'ALL': 0.5726461890753668, 2: 0.0, 1: 0.57290...","{'ALL': 0.3291705278506425, 2: 0.0, 1: 0.32914...","{'ALL': 0.48700646804676906, 2: 0.0, 1: 0.5104...","{'ALL': 0.3199745679229612, 2: 0.0, 1: 0.34804...","{'ALL': 0.5493269975707225, 2: 0.0, 1: 0.53174..."
173e2b27-8bc6-44d1-b581-a1913d6b1894,"{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.2...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.3...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.2...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.4...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.3...","{5: 0.02829018646578775, 16: 0.0, 1: 0.2256331...","{5: 0.04241223439903114, 16: 0.0, 1: 0.3995683..."
2058f680-aace-4b31-aed4-f34397b75096,"{'ALL': 0.0, 1: 0.0}","{'ALL': 0.0, 1: 0.0}","{'ALL': 0.0, 1: 0.0}","{'ALL': 0.0, 1: 0.0}","{'ALL': 0.0, 1: 0.0}","{'ALL': 0.0, 1: 0.0}","{'ALL': 0.0, 1: 0.0}"
22fde42e-45d7-46cd-9d6f-3a4dbabbc579,"{'ALL': 0.8513695392006424, 1: 0.0, 5: 0.0, 6:...","{'ALL': 0.9278731650522143, 1: 0.0, 5: 0.0, 6:...","{'ALL': 1.4550731765156883, 1: 0.0, 5: 0.0, 6:...","{'ALL': 1.6008849858280871, 1: 0.0, 5: 0.0, 6:...","{'ALL': 1.1150348133262886, 1: 0.0, 5: 0.0, 6:...","{6: 0.0, 'ALL': 0.8159645645212904, 5: 0.0, 1:...","{6: 0.0, 'ALL': 1.2074720313003113, 5: 0.0, 1:..."
3dc15e4b-b60d-4c49-aae8-97cf9664af51,"{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}"
423a069b-27b8-44ed-9ef3-ca3cf9470970,"{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}"
457bce7b-9300-4c10-9a97-070b3c0d081d,"{'ALL': 0.0, 3: 0.0, 1: 0.0, 5: 0.0}","{'ALL': 0.0, 3: 0.0, 1: 0.0, 5: 0.0}","{'ALL': 0.0, 3: 0.0, 1: 0.0, 5: 0.0}","{'ALL': 0.0, 3: 0.0, 1: 0.0, 5: 0.0}","{'ALL': 0.0, 3: 0.0, 1: 0.0, 5: 0.0}","{'ALL': 0.0, 5: 0.0, 1: 0.0, 3: 0.0}","{'ALL': 0.0, 5: 0.0, 1: 0.0, 3: 0.0}"
478cd985-f4fc-45b4-ac71-bf78cf11e07b,"{4: 0.0, 1: 0.6698111844586713, 5: 0.0, 6: 0.0...","{4: 0.0, 1: 0.7836775795970184, 5: 0.0, 6: 0.0...","{4: 0.0, 1: 0.6351694588966396, 5: 0.0, 6: 0.0...","{4: 0.0, 1: 0.6698111844586713, 5: 0.0, 6: 0.0...","{4: 0.0, 1: 1.3333333333333333, 5: 0.0, 6: 0.0...","{2: 0.0, 5: 0.0, 1: 0.3751711466950555, 6: 0.0...","{2: 0.0, 5: 0.0, 1: 0.29732228838711444, 6: 0...."


In [39]:
df_scores2 = df_scores12.merge(df_scores22, how='outer', left_index=True, right_index=True, sort=False)
joblib.dump(df_scores2, 'df_scores2.pkl')
df_scores2.to_csv('df_scores2.csv', sep=';', encoding='latin1', index=True)

df_scores2

,GradientBoosting,LSTM,TBATS,Ensemble,Chronos,"amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)",MEDIAN_MAD_60
05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,"{6: 0.2687836755945146, 3: 0.3791340371395408,...","{6: 0.2840297365726457, 3: 0.39831999423898784...","{6: 0.28887472015507765, 3: 0.3998031356637399...","{6: 0.2687836755945146, 3: 0.37913403603376566...","{6: 0.27765032825534464, 3: 0.4106718769057024...","{6: 0.30689680175455675, 'ALL': 0.744088952280...","{6: 0.25892320680613623, 'ALL': 0.626835301339..."
0661e926-f219-456e-bde5-612fee6f891c,"{1: 0.9469462634855225, 'ALL': 0.9469462634855...","{1: 1.0130325631496984, 'ALL': 1.013032565065393}","{1: 0.7858184083575777, 'ALL': 0.7858184083575...","{1: 0.9469462429746577, 'ALL': 0.9469462429746...","{1: 1.056538365959226, 'ALL': 1.0913607818329885}","{'ALL': 1.1031300133220414, 1: 1.1054466393912...","{'ALL': 0.7382457467602968, 1: 0.4199060061458..."
0ad2f65b-ee0e-4d50-93e2-84b76282325f,"{'ALL': 1.041343146475989, 2: 0.0, 1: 1.041317...","{'ALL': 1.1238143559887415, 2: 0.0, 1: 1.12378...","{'ALL': 1.0120813085330855, 2: 0.0, 1: 1.01266...","{'ALL': 1.0413431464688145, 2: 0.0, 1: 1.04131...","{'ALL': 0.9269345902608628, 2: 0.0, 1: 0.94887...","{'ALL': 1.0601233821947162, 2: 0.0, 1: 1.14345...","{'ALL': 0.9737400510003915, 2: 0.0, 1: 0.97456..."
173e2b27-8bc6-44d1-b581-a1913d6b1894,"{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.9...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.9...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.8...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.9...","{9: 0.0, 16: 0.0, 18: 0.0, 14: 0.0, 'ALL': 0.8...","{5: 0.09812570207682786, 16: 0.0, 1: 0.8058184...","{5: 0.0751953529461361, 16: 0.0, 1: 0.72970889..."
2058f680-aace-4b31-aed4-f34397b75096,"{'ALL': 0.0, 1: 0.0}","{'ALL': 0.0, 1: 0.0}","{'ALL': 0.0, 1: 0.0}","{'ALL': 0.0, 1: 0.0}","{'ALL': 0.0, 1: 0.0}","{'ALL': 0.0, 1: 0.0}","{'ALL': 0.0, 1: 0.0}"
22fde42e-45d7-46cd-9d6f-3a4dbabbc579,"{'ALL': 2.8923324425097823, 1: 0.0, 5: 0.0, 6:...","{'ALL': 3.1211558171566844, 1: 0.0, 5: 0.0, 6:...","{'ALL': 2.510709584656852, 1: 0.0, 5: 0.0, 6: ...","{'ALL': 2.8923329258779815, 1: 0.0, 5: 0.0, 6:...","{'ALL': 1.3937935166578606, 1: 0.0, 5: 0.0, 6:...","{6: 0.0, 'ALL': 2.8169549950638304, 5: 0.0, 1:...","{6: 0.0, 'ALL': 2.2051172231290073, 5: 0.0, 1:..."
3dc15e4b-b60d-4c49-aae8-97cf9664af51,"{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}"
423a069b-27b8-44ed-9ef3-ca3cf9470970,"{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}"
457bce7b-9300-4c10-9a97-070b3c0d081d,"{'ALL': 0.0, 3: 0.0, 1: 0.0, 5: 0.0}","{'ALL': 0.0, 3: 0.0, 1: 0.0, 5: 0.0}","{'ALL': 0.0, 3: 0.0, 1: 0.0, 5: 0.0}","{'ALL': 0.0, 3: 0.0, 1: 0.0, 5: 0.0}","{'ALL': 0.0, 3: 0.0, 1: 0.0, 5: 0.0}","{'ALL': 0.0, 5: 0.0, 1: 0.0, 3: 0.0}","{'ALL': 0.0, 5: 0.0, 1: 0.0, 3: 0.0}"
478cd985-f4fc-45b4-ac71-bf78cf11e07b,"{4: 0.0, 1: 1.238696684724528, 5: 0.0, 6: 0.0,...","{4: 0.0, 1: 1.3388879677093661, 5: 0.0, 6: 0.0...","{4: 0.0, 1: 1.1215817878002738, 5: 0.0, 6: 0.0...","{4: 0.0, 1: 1.238696684724528, 5: 0.0, 6: 0.0,...","{4: 0.0, 1: 1.6666666666666665, 5: 0.0, 6: 0.0...","{2: 0.0, 5: 0.0, 1: 1.2863757285144768, 6: 0.0...","{2: 0.0, 5: 0.0, 1: 1.0571014171848696, 6: 0.0..."


In [40]:
df_scores1.to_csv('scores1_per_channel.csv', sep=';', encoding='latin1', index=True)
df_scores2.to_csv('scores2_per_channel.csv', sep=';', encoding='latin1', index=True)